### https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/determine-eval#evaluation-instance-inputs

In [1]:
import pandas as pd

# Example DataFrame with prompts and ground truth references
prompts_df = pd.DataFrame(
    {
        "prompt": ["What is 2+2?", "What is 3+3?"],
        "response": ["4", "5"],
        "reference": ["4", "6"],
    }
)

In [2]:
from vertexai import Client, types
import render_util
import google.auth
from google.genai import types as genai_types
httpOptions = genai_types.HttpOptions(
    retry_options=genai_types.HttpRetryOptions(
        attempts=5,           # 최대 재시도 횟수
        initial_delay=1.0,    # 첫 대기 시간
        http_status_codes=[429, 500, 502, 503, 504] # 재시도 대상 에러 코드
    )
)
_, PROJECT_ID = google.auth.default()
LOCATION = "global"
client = Client(project=PROJECT_ID, location=LOCATION)

In [3]:
code_snippet = """
def evaluate(instance):
    if instance['response'] == instance['reference']:
        return 1.0
    return 0.0
"""

def evaluate(instance):
    def get_text_from_part(target_dict):
        try:
            return target_dict['parts'][0]['text']
        except (KeyError, IndexError, TypeError):
            return None
    ref_text = get_text_from_part(instance.get('reference', {}).get('response', {}))
    res_text = get_text_from_part(instance.get('response', {}))
    if ref_text == res_text:
        return 1.0
    return 0.0

custom_metric = types.Metric(
    name="my_custom_code_metric",
    #remote_custom_function=code_snippet,
    custom_function=evaluate
)

evaluation_result = client.evals.evaluate(
    dataset=prompts_df,
    metrics=[custom_metric],
)

C:\Users\jeehyeok\AppData\Roaming\Python\Python312\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey
Computing Metrics for Evaluation Dataset: 100%|██████████| 2/2 [00:00<00:00, 18.30it/s]


In [4]:
render_util.display_evaluation_result(evaluation_result)

In [ ]:
# Input original prompt to optimize
prompt = """You are a professional chef. Your goal is teaching how to cook healthy cooking recipes to your apprentice.

Given a question from your apprentice and some context, provide the correct answer to the question.
Use the context to return a single and correct answer with some explanation.
"""

# Optimize prompt
output = client.prompts.optimize(
    prompt=prompt,
    config=types.OptimizeConfig(
            #optimization_target=types.OptimizeTarget.OPTIMIZATION_TARGET_GEMINI_NANO #for smaller model
        )
    )

# View optimized prompt
print(output.model_dump_json(indent=2))

{
  "raw_text_response": "```json\n{\n  \"optimization_type\": \"zero_shot_prompt_optimization\",\n  \"applicable_guidelines\": [\n    {\n      \"applicable_guideline\": \"Underspecified\",\n      \"suggested_improvement\": \"The task is too high-level. The suggested prompt provides specific instructions on how to formulate the answer and explanation, and what to do if the answer is not in the context.\",\n      \"text_before_change\": \"Given a question from your apprentice and some context, provide the correct answer to the question.\\nUse the context to return a single and correct answer with some explanation.\",\n      \"text_after_change\": \"### Instructions\\n1.  Carefully read the apprentice's `question` and the `context`.\\n2.  Formulate a clear, single answer to the question using only information found in the context.\\n3.  After the answer, provide a brief explanation with practical cooking tips or health benefits related to the answer.\\n4.  If the context does not contain

In [9]:
system_instructions = """
You are an AI assistant skilled in analyzing articles. 
Your task is to extract key arguments, evaluate evidence quality, identify potential biases, 
and summarize core findings into a concise, actionable, and objective report. Make sure the response is less than 50 words
"""
df = pd.DataFrame(
    {
        "prompt": [
            """
            A new study claims that eating chocolate daily improves cognitive function by 20%. 
            The research, funded by a leading chocolate manufacturer, surveyed 50 chocolate lovers who self-reported feeling sharper. 
            However, there was no control group, and the self-reporting method is highly subjective.
            """, 
            """
            Our city's public transit expansion plan will reduce traffic congestion by 30%% and lower emissions. 
            The data is based on a comprehensive 5-year urban traffic simulation study conducted by independent traffic engineers. 
            No corporate sponsorships were involved, though the mayor heavily champions this to secure re-election.
            """],
        "model_response": [
            """
            The article claims daily chocolate improves cognition by 20%. 
            However, the evidence is weak, relying on self-reporting from 50 participants without a control group. 
            Funding by a chocolate manufacturer introduces strong commercial bias. 
            Thus, the findings are highly subjective and unreliable for cognitive enhancement strategies.
            """, 
            """
            The transit plan claims to cut traffic by 30%% and reduce emissions. 
            Supported by a 5-year independent simulation, the evidence is strong. 
            While the mayor's re-election campaign introduces political bias, the data remains credible. 
            Recommendation: Proceed with the expansion due to solid supporting evidence.
            """],
        "target_response": [
            """
            Daily chocolate boosts cognition. 
            Evidence: Weak; 50 self-reported participants, no control group. 
            Bias: High; funded by chocolate manufacturer. 
            Finding: The claim is unreliable due to conflict of interest and poor methodology. 
            Action: Do not base health dietary decisions on this study.
            """, 
            """
            Transit expansion reduces traffic 30%% and emissions. 
            Evidence: Strong; 5-year independent simulation. 
            Bias: Low; independent, though political motivation exists. 
            Finding: Highly credible benefits. 
            Action: Approve and proceed with the public transit expansion plan based on robust data.
            """],
    }
)
config = types.OptimizeConfig(
    optimization_target=types.OptimizeTarget.OPTIMIZATION_TARGET_FEW_SHOT_TARGET_RESPONSE,
    examples_dataframe=df,
)
response = client.prompts.optimize(
    prompt=system_instructions,
    config=config,
)

print(response.parsed_response.suggested_prompt)

You are an AI assistant skilled in analyzing articles. Your task is to provide a concise, actionable, and objective report that follows a strict format.

**Structure and Formatting Guidelines:**
- The response must be structured with each of the following components on a new line.
- **Line 1:** A one-sentence summary of the article's core argument. Do not use a label for this line.
- **Subsequent Lines:** Use the following labels exactly as written, followed by a colon:
  - `Evidence:`
  - `Bias:`
  - `Finding:`
  - `Action:`

**Constraints:**
- The entire response must be less than 50 words.
- Be extremely direct and use sentence fragments to maintain brevity.


### https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/prompts/data-driven-optimizer